# DATA ANALYSIS 
## DATA COVERAGE 
### 1. DATA CLEANING


#### b. WB CWON

In [3]:
# ============================================================
# 1. Paths and settings
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# Input files
WB_FINAL_PATH = "WB_final.csv"

# If you already downloaded the CWON file, use this local path.
# Otherwise, the code will download it from the World Bank URL.
CWON_LOCAL_PATH = "WB_CWON_WIDEF.csv"
CWON_URL = "https://data360files.worldbank.org/data360-data/data/WB_CWON/WB_CWON_WIDEF.csv"

# Output file
OUTPUT_PATH = "WB_final_CWON_full_indicator_variations.csv"

# Number of latest reporting years to check.
# Example: if the latest columns are 2019 and 2020, a country gets 1 if either year has data.
N_LATEST_YEARS = 2

In [4]:
# ============================================================
# 2. Load datasets
# ============================================================

# Load WB_final template
wb_final = pd.read_csv(WB_FINAL_PATH)

# Load CWON data from local file if available, otherwise from URL
if Path(CWON_LOCAL_PATH).exists():
    cwon = pd.read_csv(CWON_LOCAL_PATH)
else:
    cwon = pd.read_csv(CWON_URL)
    cwon.to_csv(CWON_LOCAL_PATH, index=False)

print("WB_final shape:")
print(wb_final.shape)

print("CWON shape:")
print(cwon.shape)

print("WB_final preview:")
print(wb_final.head())

print("CWON preview:")
print(cwon.head())

WB_final shape:
(0, 223)
CWON shape:
(8350, 48)
WB_final preview:
Empty DataFrame
Columns: [no, indicator, Definition, unit, source, Afghanistan, Albania, Algeria, American Samoa, Andorra, Angola, Antigua and Barbuda, Argentina, Armenia, Aruba, Australia, Austria, Azerbaijan, Bahamas, Bahrain, Bangladesh, Barbados, Belarus, Belgium, Belize, Benin, Bermuda, Bhutan, Bolivia, Bosnia and Herzegovina, Botswana, Brazil, British Virgin Islands, Brunei Darussalam, Bulgaria, Burkina Faso, Burundi, Cabo Verde, Cambodia, Cameroon, Canada, Cayman Islands, Central African Republic, Chad, Channel Islands, Chile, China, Colombia, Comoros, Congo Dem. Rep., Congo Rep., Costa Rica, Côte dIvoire, Croatia, Cuba, Curaçao, Cyprus, Czech Republic, Denmark, Djibouti, Dominica, Dominican Republic, Ecuador, Egypt, El Salvador, Equatorial Guinea, Eritrea, Estonia, Eswatini, Ethiopia, Faroe Islands, Fiji, Finland, France, French Polynesia, Gabon, Gambia, Georgia, Germany, Ghana, Gibraltar, Greece, Greenland, Gren

In [5]:
# ============================================================
# 3. Identify template country columns
# ============================================================

# The first five columns in WB_final are assumed to be:
# no, indicator, Definition, unit, source
base_columns = ["no", "indicator", "Definition", "unit", "source"]

# Country columns start after the first five columns
country_cols = list(wb_final.columns[5:])

print("Number of country columns in WB_final:")
print(len(country_cols))

print("First 10 country columns:")
print(country_cols[:10])

Number of country columns in WB_final:
218
First 10 country columns:
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba']


In [6]:
# ============================================================
# 4. Identify latest reporting years in CWON
# ============================================================

# Year columns are columns whose names are all digits, e.g. 1995, 1996, ..., 2020
year_cols = [col for col in cwon.columns if str(col).isdigit()]
year_cols = sorted(year_cols, key=lambda x: int(x))

# Use the latest N years
latest_year_cols = year_cols[-N_LATEST_YEARS:]

print("All year columns:")
print(year_cols)

print("Latest reporting years used for availability:")
print(latest_year_cols)

All year columns:
['1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020']
Latest reporting years used for availability:
['2019', '2020']


In [7]:
# ============================================================
# 5. Normalize country names to match WB_final
# ============================================================

# CWON country names may differ slightly from WB_final country names.
# This mapping converts CWON names into WB_final names.
country_name_map = {
    "Bahamas, The": "Bahamas",
    "Congo, Dem. Rep.": "Congo Dem. Rep.",
    "Congo, Rep.": "Congo Rep.",
    "Cote d'Ivoire": "Côte dIvoire",
    "Egypt, Arab Rep.": "Egypt",
    "Gambia, The": "Gambia",
    "Hong Kong SAR, China": "Hong Kong (S.A.R.)",
    "Iran, Islamic Rep.": "Iran",
    "Korea, Dem. People's Rep.": "Korea Dem. Peoples Rep.",
    "Korea, Rep.": "South Korea",
    "Macao SAR, China": "Macao SAR",
    "Micronesia, Fed. Sts.": "Micronesia",
    "Russian Federation": "Russia",
    "Sao Tome and Principe": "São Tomé and Principe",
    "Turkiye": "Turkey",
    "Venezuela, RB": "Venezuela",
    "Yemen, Rep.": "Yemen"
}

# Create a working copy of CWON
work_df = cwon.copy()

# Apply country-name mapping
work_df["template_country"] = work_df["REF_AREA_LABEL"].replace(country_name_map)

# Keep only countries that exist in WB_final
work_df = work_df[work_df["template_country"].isin(country_cols)].copy()

print("Number of CWON rows after matching countries to WB_final:")
print(work_df.shape[0])

print("Number of matched countries:")
print(work_df["template_country"].nunique())

Number of CWON rows after matching countries to WB_final:
8294
Number of matched countries:
150


In [8]:
# ============================================================
# 6. Define the full CWON indicator variation
# ============================================================

# The revised indicator definition is the combination of:
# INDICATOR_LABEL + SEX_LABEL + COMP_BREAKDOWN_1_LABEL + COMP_BREAKDOWN_2_LABEL

indicator_component_cols = [
    "INDICATOR_LABEL",
    "SEX_LABEL",
    "COMP_BREAKDOWN_1_LABEL",
    "COMP_BREAKDOWN_2_LABEL"
]

metadata_cols = [
    "INDICATOR_LABEL",
    "SEX_LABEL",
    "COMP_BREAKDOWN_1_LABEL",
    "COMP_BREAKDOWN_2_LABEL",
    "UNIT_MEASURE_LABEL",
    "DATABASE_ID_LABEL"
]

# Fill missing metadata values so grouping and labels work cleanly
for col in metadata_cols:
    work_df[col] = work_df[col].fillna("Not Available")

# Create combined indicator name
work_df["combined_indicator"] = (
    work_df["INDICATOR_LABEL"] + " | " +
    work_df["SEX_LABEL"] + " | " +
    work_df["COMP_BREAKDOWN_1_LABEL"] + " | " +
    work_df["COMP_BREAKDOWN_2_LABEL"]
)

# Create definition field
work_df["definition_text"] = (
    "SEX_LABEL: " + work_df["SEX_LABEL"] + "; " +
    "COMP_BREAKDOWN_1_LABEL: " + work_df["COMP_BREAKDOWN_1_LABEL"] + "; " +
    "COMP_BREAKDOWN_2_LABEL: " + work_df["COMP_BREAKDOWN_2_LABEL"] + "; " +
    "DATABASE_ID_LABEL: " + work_df["DATABASE_ID_LABEL"]
)

print("Number of unique full indicator variations:")
print(work_df["combined_indicator"].nunique())

print("Example full indicators:")
print(work_df[["combined_indicator", "definition_text", "UNIT_MEASURE_LABEL", "DATABASE_ID_LABEL"]].drop_duplicates().head(10))

Number of unique full indicator variations:
76
Example full indicators:
                                  combined_indicator  \
0  Domestic comprehensive wealth index | Total | ...   
1  Domestic comprehensive wealth index | Total | ...   
2  Human capital | Female | Aggregation: per capi...   
3  Human capital | Female | Aggregation: total | ...   
4  Human capital | Male | Aggregation: per capita...   
5  Human capital | Male | Aggregation: total | No...   
6  Human capital | Total | Aggregation: per capit...   
7  Human capital | Total | Aggregation: total | N...   
8  Nonrenewable natural capital | Total | Aggrega...   
9  Nonrenewable natural capital | Total | Aggrega...   

                                     definition_text UNIT_MEASURE_LABEL  \
0  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   
1  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   
2  SEX_LABEL: Female; COMP_BREAKDOWN_1_LABEL: Agg...         US dollars   
3  SEX_LABE

In [9]:
# ============================================================
# 7. Calculate country availability for latest reporting years
# ============================================================

# Convert latest year columns to numeric
for col in latest_year_cols:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

# Availability equals 1 if at least one of the latest reporting years has data
work_df["available_latest"] = work_df[latest_year_cols].notna().any(axis=1).astype(int)

# Group by the full indicator variation and country
group_cols = [
    "combined_indicator",
    "definition_text",
    "UNIT_MEASURE_LABEL",
    "DATABASE_ID_LABEL",
    "template_country"
]

availability_long = (
    work_df
    .groupby(group_cols, dropna=False)["available_latest"]
    .max()
    .reset_index()
)

print("Long availability table shape:")
print(availability_long.shape)

print("Long availability preview:")
print(availability_long.head())

Long availability table shape:
(8294, 6)
Long availability preview:
                                  combined_indicator  \
0  Domestic comprehensive wealth index | Total | ...   
1  Domestic comprehensive wealth index | Total | ...   
2  Domestic comprehensive wealth index | Total | ...   
3  Domestic comprehensive wealth index | Total | ...   
4  Domestic comprehensive wealth index | Total | ...   

                                     definition_text UNIT_MEASURE_LABEL  \
0  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   
1  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   
2  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   
3  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   
4  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...         US dollars   

                       DATABASE_ID_LABEL template_country  available_latest  
0  The Changing Wealth of Nations (CWON)          Albania         

In [10]:
# ============================================================
# 8. Pivot availability to country-wide format
# ============================================================

availability_pivot = availability_long.pivot_table(
    index=[
        "combined_indicator",
        "definition_text",
        "UNIT_MEASURE_LABEL",
        "DATABASE_ID_LABEL"
    ],
    columns="template_country",
    values="available_latest",
    aggfunc="max",
    fill_value=0
).reset_index()

# Make sure every country from WB_final exists as a column
for country in country_cols:
    if country not in availability_pivot.columns:
        availability_pivot[country] = 0

print("Wide availability table shape:")
print(availability_pivot.shape)

print("Wide availability preview:")
print(availability_pivot.head())

Wide availability table shape:
(76, 222)
Wide availability preview:
template_country                                 combined_indicator  \
0                 Domestic comprehensive wealth index | Total | ...   
1                 Domestic comprehensive wealth index | Total | ...   
2                 Foreign assets | Total | Aggregation: per capi...   
3                 Foreign assets | Total | Aggregation: total | ...   
4                 Foreign liabilities | Total | Aggregation: per...   

template_country                                    definition_text  \
0                 SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...   
1                 SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...   
2                 SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...   
3                 SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...   
4                 SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...   

template_country UNIT_MEASURE_LABEL                      DATABASE_ID_LABEL  \


In [11]:
# ============================================================
# 9. Fill WB_final template structure
# ============================================================

# Create final output dataframe using the same column structure as WB_final
final_df = pd.DataFrame()

final_df["no"] = range(1, len(availability_pivot) + 1)
final_df["indicator"] = availability_pivot["combined_indicator"]
final_df["Definition"] = availability_pivot["definition_text"]
final_df["unit"] = availability_pivot["UNIT_MEASURE_LABEL"]
final_df["source"] = availability_pivot["DATABASE_ID_LABEL"]

# Add country availability columns
for country in country_cols:
    final_df[country] = availability_pivot[country].astype(int)

# Reorder to exactly match WB_final columns
final_df = final_df[wb_final.columns]

print("Final output shape:")
print(final_df.shape)

print("Final output preview:")
print(final_df.head())

Final output shape:
(76, 223)
Final output preview:
   no                                          indicator  \
0   1  Domestic comprehensive wealth index | Total | ...   
1   2  Domestic comprehensive wealth index | Total | ...   
2   3  Foreign assets | Total | Aggregation: per capi...   
3   4  Foreign assets | Total | Aggregation: total | ...   
4   5  Foreign liabilities | Total | Aggregation: per...   

                                          Definition        unit  \
0  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   
1  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   
2  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   
3  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   
4  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   

                                  source  Afghanistan  Albania  Algeria  \
0  The Changing Wealth of Nations (CWON)            0        1        0   
1  The Changing Wealth of Na

C:\Users\landa\AppData\Local\Temp\ipykernel_25528\263948466.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[country] = availability_pivot[country].astype(int)
C:\Users\landa\AppData\Local\Temp\ipykernel_25528\263948466.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df[country] = availability_pivot[country].astype(int)
C:\Users\landa\AppData\Local\Temp\ipykernel_25528\263948466.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, whi

In [12]:
# ============================================================
# 10. Validation checks
# ============================================================

# Check example requested by user: Angola availability for Domestic comprehensive wealth index
angola_check = final_df.loc[
    final_df["indicator"].str.contains("Domestic comprehensive wealth index", case=False, na=False),
    ["no", "indicator", "Definition", "unit", "source", "Angola"]
]

print("Angola check for Domestic comprehensive wealth index:")
print(angola_check)

# Check how many countries have availability for each indicator
country_availability_counts = final_df[country_cols].sum(axis=1)

summary_check = final_df[["no", "indicator", "unit", "source"]].copy()
summary_check["available_country_count"] = country_availability_counts

print("Availability count summary:")
print(summary_check.head(20))

print("Minimum available countries for an indicator:")
print(summary_check["available_country_count"].min())

print("Maximum available countries for an indicator:")
print(summary_check["available_country_count"].max())

Angola check for Domestic comprehensive wealth index:
   no                                          indicator  \
0   1  Domestic comprehensive wealth index | Total | ...   
1   2  Domestic comprehensive wealth index | Total | ...   

                                          Definition        unit  \
0  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   
1  SEX_LABEL: Total; COMP_BREAKDOWN_1_LABEL: Aggr...  US dollars   

                                  source  Angola  
0  The Changing Wealth of Nations (CWON)       1  
1  The Changing Wealth of Nations (CWON)       1  
Availability count summary:
    no                                          indicator        unit  \
0    1  Domestic comprehensive wealth index | Total | ...  US dollars   
1    2  Domestic comprehensive wealth index | Total | ...  US dollars   
2    3  Foreign assets | Total | Aggregation: per capi...  US dollars   
3    4  Foreign assets | Total | Aggregation: total | ...  US dollars   
4    5  Foreig

In [13]:
# ============================================================
# 11. Save final file
# ============================================================

final_df.to_csv(OUTPUT_PATH, index=False)

print("Saved completed file:")
print(OUTPUT_PATH)

print("Final saved shape:")
print(final_df.shape)

Saved completed file:
WB_final_CWON_full_indicator_variations.csv
Final saved shape:
(76, 223)
